In [1]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")
client = Groq(api_key=os.getenv("GROQ_API_KEY")) 

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Объясни одним предложением, что такое TF-IDF."}
    ]
)

print(response.choices[0].message.content)

TF‑IDF (term frequency‑inverse document frequency) — это статистическая мера, которая оценивает важность слова в документе, умножая частоту его появления в данном документе (TF) на логарифм обратного отношения общего числа документов к числу документов, содержащих это слово (IDF).


In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import sys
sys.path.append("..")
from src.text_cleaning import clean_text
import os
import numpy as np

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
df = pd.read_csv('../data/eval/Resume.csv')
df['clean_text'] = df['Resume_str'].apply(clean_text)

In [4]:

EMB_PATH = "../data/eval/resume_embeddings.npy"

if os.path.exists(EMB_PATH):
    # файл уже есть: просто загружаем, это быстро
    resume_embeddings = np.load(EMB_PATH)
else:
    # файла нет: считаем один раз и сохраняем
    resume_embeddings = model.encode(df['clean_text'].tolist(), show_progress_bar=True)
    np.save(EMB_PATH, resume_embeddings)


In [5]:

def search(vacancy_text, k=5):
    vacancy_clean = clean_text(vacancy_text)
    vacancy_vec = model.encode([vacancy_clean])
    scores = cosine_similarity(vacancy_vec, resume_embeddings)[0]
    result = df.copy()
    result['score'] = scores
    return result.sort_values('score', ascending=False).head(k)

search("experienced accountant with tax and bookkeeping skills")[['Category', 'score']]

,Category,score
1833,ACCOUNTANT,0.773654
1908,ACCOUNTANT,0.745314
1820,ACCOUNTANT,0.736703
1923,ACCOUNTANT,0.723167
1892,ACCOUNTANT,0.699115


In [11]:
import json
from groq import RateLimitError

SYSTEM_PROMPT = """You are an experienced recruiter. You compare a job vacancy with a candidate's resume and rate how well the candidate fits.

Scoring scale:
1.0 - perfect fit: same role, all key skills and relevant experience
0.7 - good fit: same role, minor gaps
0.4 - partial fit: related field, but important requirements are missing
0.1 - weak fit: different profession, only a few overlapping skills
0.0 - no relation at all

Be strict: do not give 0.7 or higher unless most key requirements are met.
Reply ONLY with JSON: {"score": <number from 0 to 1>, "explanation": "<2-3 sentences>"}
Write the explanation in the same language as the vacancy."""



def llm_score(vacancy_text, resume_text, retries=5):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"VACANCY:\n{vacancy_text}\n\nRESUME:\n{resume_text[:4000]}"},
                ],
                response_format={"type": "json_object"},
                temperature=0,
            )
            return json.loads(response.choices[0].message.content)
        except RateLimitError:
            print("Лимит, жду 20 секунд...")
            time.sleep(20)
    raise RuntimeError("Не удалось получить ответ от LLM: лимит исчерпан")

In [7]:
vacancies = {
    "ACCOUNTANT": "Accountant to manage general ledger, accounts payable and receivable, tax returns, reconciliations and monthly financial reporting.",
    "ADVOCATE": "Attorney to provide legal advice, draft contracts, conduct legal research and represent clients in court and litigation.",
    "AGRICULTURE": "Agriculture specialist to manage crop production, farm operations, soil and irrigation, livestock and agronomy programs.",
    "APPAREL": "Apparel merchandiser to manage clothing collections, garment production, fashion trends, fabric sourcing and retail buying.",
    "ARTS": "Arts program coordinator to organize exhibitions, theater and music events, work with artists and manage art education programs.",
    "AUTOMOBILE": "Automotive technician to diagnose and repair vehicles, engines, brakes and transmissions, and perform vehicle maintenance.",
    "AVIATION": "Aviation maintenance technician to inspect and repair aircraft, follow FAA regulations and support flight operations.",
    "BANKING": "Banking officer to open customer accounts, process loans and deposits, ensure compliance and provide branch customer service.",
    "BPO": "BPO call center agent to handle inbound and outbound customer calls, resolve client issues and meet process quality targets.",
    "BUSINESS-DEVELOPMENT": "Business development manager to find new clients, build partnerships, generate leads and grow company revenue.",
    "CHEF": "Chef to prepare dishes, plan menus, manage the kitchen staff, ensure food safety and control food costs.",
    "CONSTRUCTION": "Construction project manager to supervise building sites, contractors, safety, schedules and construction budgets.",
    "CONSULTANT": "Consultant to analyze client business problems, recommend solutions, lead projects and present findings to stakeholders.",
    "DESIGNER": "Designer to create visual concepts, layouts and branding using Adobe Photoshop, Illustrator and InDesign.",
    "DIGITAL-MEDIA": "Digital media specialist to manage social media, create video and web content, run online campaigns and analyze engagement.",
    "ENGINEERING": "Engineer to design mechanical and electrical systems, use AutoCAD, perform testing and manage engineering projects.",
    "FINANCE": "Financial analyst to build financial models, forecasts and budgets, analyze investments and prepare financial statements.",
    "FITNESS": "Fitness trainer to lead personal training sessions and group exercise classes, design workout programs and coach clients.",
    "HEALTHCARE": "Healthcare professional to provide patient care, maintain medical records, assist physicians and follow clinical procedures.",
    "HR": "HR specialist to manage recruitment, onboarding, employee relations, payroll, benefits and HR policies.",
    "INFORMATION-TECHNOLOGY": "IT specialist to support networks, servers and help desk, manage systems administration and troubleshoot hardware and software.",
    "PUBLIC-RELATIONS": "Public relations specialist to write press releases, manage media relations, plan events and handle corporate communications.",
    "SALES": "Sales representative to sell products, meet sales targets, manage customer accounts and negotiate deals.",
    "TEACHER": "Teacher to plan lessons, teach students in the classroom, assess student progress and communicate with parents.",
}



In [8]:

chef_vacancy = vacancies["CHEF"]   # или напиши текст вакансии повара заново

top = search(chef_vacancy, k=1)
good_resume = top['Resume_str'].iloc[0]
print(llm_score(chef_vacancy, good_resume))

bad_resume = df[df['Category'] == 'ACCOUNTANT']['Resume_str'].iloc[0]
print(llm_score(chef_vacancy, bad_resume))

{'score': 1.0, 'explanation': 'The candidate has extensive chef experience, proven menu planning, kitchen staff management, ServSafe food safety certification, and strong cost‑control expertise, fully matching the vacancy requirements.'}
{'score': 0.0, 'explanation': "The candidate's experience is entirely in accounting and finance, with no culinary training, kitchen management, or food safety knowledge. Therefore there is no relevant overlap with the chef role requirements."}


In [9]:
acc_vacancy = vacancies["ACCOUNTANT"]   # или напиши текст вакансии повара заново

top = search(acc_vacancy, k=1)
good_resume = top['Resume_str'].iloc[0]
print(llm_score(acc_vacancy, good_resume))

bad_resume = df[df['Category'] == 'TEACHER']['Resume_str'].iloc[0]
print(llm_score(acc_vacancy, bad_resume))

{'score': 0.7, 'explanation': 'The candidate has solid experience in general ledger, accounts payable/receivable, reconciliations and monthly financial reporting, matching the core duties of the vacancy. However, the resume does not mention tax return preparation, which is a required responsibility, resulting in a good but not perfect fit.'}
{'score': 0.1, 'explanation': "The candidate's background is primarily in teaching and executive advisory roles, with limited mention of accounting tasks. While they list some financial analysis and Excel skills, they lack experience with general ledger, AP/AR, tax returns, reconciliations, and monthly reporting required for the accountant position."}


In [12]:
def rerank(vacancy_text, k=5, shortlist=10):
    candidates = search(vacancy_text, k=shortlist)              # 1. найти топ-shortlist через search
    llm_scores = []
    explanations = []
    for resume in candidates['Resume_str']:
        result = llm_score(vacancy_text, resume)                          # 2. оценить резюме через LLM
        llm_scores.append(result['score'])
        explanations.append(result['explanation'])
    candidates['llm_score'] = llm_scores
    candidates['explanation'] = explanations
    return candidates.sort_values(['llm_score', 'score'], ascending=False).head(k)
# rerank(vacancies["ACCOUNTANT"])[['Category', 'score', 'llm_score', 'explanation']]

In [ ]:
# results = []

# for category, text in vacancies.items():
#     top = rerank(text)
#     precision = (top['Category'] == category).mean()
#     results.append({"category": category, "precision@5": precision})

# results_df = pd.DataFrame(results)
# results_df

# print(results_df['precision@5'].mean())



RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m7qeqss4evybraeq3705cwzf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199032, Requested 1557. Please try again in 4m14.448s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# print("Готово вакансий:", len(results))
# partial_df = pd.DataFrame(results)
# partial_df.to_csv("../data/eval/llm_results_partial.csv", index=False)
# partial_df

Готово вакансий: 16


,category,precision@5
0,ACCOUNTANT,1.0
1,ADVOCATE,0.2
2,AGRICULTURE,1.0
3,APPAREL,0.6
4,ARTS,1.0
5,AUTOMOBILE,0.0
6,AVIATION,1.0
7,BANKING,1.0
8,BPO,0.0
9,BUSINESS-DEVELOPMENT,1.0


In [13]:
results = pd.read_csv("../data/eval/llm_results_partial.csv").to_dict("records")
print("Уже готово:", len(results))

done = {r["category"] for r in results}

for category, text in vacancies.items():
    if category in done:
        continue
    top = rerank(text)
    precision = (top['Category'] == category).mean()
    results.append({"category": category, "precision@5": precision})
    pd.DataFrame(results).to_csv("../data/eval/llm_results_partial.csv", index=False)

results_df = pd.DataFrame(results)
print("Готово:", len(results_df), "из 24")
print(results_df['precision@5'].mean())



Уже готово: 16
Готово: 24 из 24
0.775


Все три модели из задания готовы